In [27]:
import pandas as pd
import numpy as np

In [28]:
stats = pd.read_csv("../data/raw/player_stats_fryzigg_2024.csv")

In [29]:
stats.head()

,venue_name,match_id,match_home_team,match_away_team,match_date,match_local_time,match_attendance,match_round,match_home_team_goals,match_home_team_behinds,...,marks_on_lead,pressure_acts,rating_points,ruck_contests,score_launches,shots_at_goal,spoils,subbed,player_position,date
0,SCG,16586,Sydney,Melbourne,2024-03-07,19:30:00,0,Opening Round,12,14,...,0,8,11.1,0,0,0,8,NaN,FB,2024-03-07
1,SCG,16586,Sydney,Melbourne,2024-03-07,19:30:00,0,Opening Round,12,14,...,0,11,5.5,89,3,2,0,NaN,RK,2024-03-07
2,SCG,16586,Sydney,Melbourne,2024-03-07,19:30:00,0,Opening Round,12,14,...,0,8,11.2,0,1,0,9,NaN,CHB,2024-03-07
3,SCG,16586,Sydney,Melbourne,2024-03-07,19:30:00,0,Opening Round,12,14,...,0,15,10.1,0,0,1,4,NaN,INT,2024-03-07
4,SCG,16586,Sydney,Melbourne,2024-03-07,19:30:00,0,Opening Round,12,14,...,0,5,12.3,0,0,0,4,NaN,BPR,2024-03-07


In [30]:
stats.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9936 entries, 0 to 9935
Data columns (total 81 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   venue_name                      9936 non-null   object 
 1   match_id                        9936 non-null   int64  
 2   match_home_team                 9936 non-null   object 
 3   match_away_team                 9936 non-null   object 
 4   match_date                      9936 non-null   object 
 5   match_local_time                9936 non-null   object 
 6   match_attendance                9936 non-null   int64  
 7   match_round                     9936 non-null   object 
 8   match_home_team_goals           9936 non-null   int64  
 9   match_home_team_behinds         9936 non-null   int64  
 10  match_home_team_score           9936 non-null   int64  
 11  match_away_team_goals           9936 non-null   int64  
 12  match_away_team_behinds         99

In [31]:
stats["player_name"] = (
    stats["player_first_name"].astype(str).str.strip()
    + " "
    + stats["player_last_name"].astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip()

# 1. Player stats 
main stats: disposals, marks, tackles, goals etc.

## 1.1. Deal with missing data

In [32]:
missing = stats.isna().sum()
missing[missing > 0]

player_is_retired    3932
supercoach_score     9936
subbed               9936
dtype: int64

In [33]:
stats[['player_is_retired', 'supercoach_score', 'subbed']].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9936 entries, 0 to 9935
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   player_is_retired  6004 non-null   object 
 1   supercoach_score   0 non-null      float64
 2   subbed             0 non-null      float64
dtypes: float64(2), object(1)
memory usage: 233.0+ KB


### `player_is_retired`
- **Description**: 3932 missing values. This column indicates whether a player has retired or not.
- **Approach**: Fill with False since players that are active enough to have match records are likely not retired, and the missing data just means the data was never populated.

In [34]:
stats.fillna({'player_is_retired': False}, inplace=True)

C:\Users\s223128143\AppData\Local\Temp\ipykernel_22112\825166848.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  stats.fillna({'player_is_retired': False}, inplace=True)


In [35]:
stats['player_is_retired'].isna().sum()

np.int64(0)

### `supercoach_score`
- **Description**: 9936 missing values. We already have `afl_fantasy_score` and `rating_points` as scoring benchmarks. Also, too many missing values. I will just drop this.

In [36]:
stats.drop('supercoach_score', inplace=True, axis=1)

### `subbed`
- **Description**: Too many missing values. I will just drop this.

In [37]:
stats.drop('subbed', inplace=True, axis=1)

## 1.2 Drop/Consolidate Redundant Columns
- Drop `date` — duplicate of `match_date`
- Drop `player_height_cm`, `player_weight_kg`, `guernsey_number`, `player_is_retired` — not relevant to per-match performance scoring

In [38]:
stats.drop(['date', 'player_height_cm', 'player_weight_kg', 'guernsey_number', 'player_is_retired'], inplace=True, axis=1)

## 1.3 Encode Categorical Variables
| Column             | Action                                                                  |
|--------------------|-------------------------------------------------------------------------|
| `player_position`    | Label-encode or one-hot encode — critical for position-adjusted weights |
| `match_winner`       | Binary flag: `won_match = (player_team == match_winner)`                  |
| `match_weather_type` | One-hot encode (Clear, Overcast, Rain, etc.)                            |
| `match_round`        | Keep as integer (ordinal); useful for time-series splits                |


### `player_position` → position group label

In [39]:
stats.player_position.value_counts()

player_position
INT     1728
FB       432
RK       432
CHB      432
BPR      432
HFFL     432
SUB      432
HBFR     432
RR       432
FPL      432
WL       432
R        432
CHF      432
C        432
FF       432
FPR      432
BPL      432
HBFL     432
HFFR     432
WR       432
Name: count, dtype: int64

In [40]:
position_map = {
    # Midfielders
    'C': 'MID', 'WL': 'MID', 'WR': 'MID', 'RR': 'MID',
    'INT': 'MID', 'R': 'MID',
    # Ruckmen
    'RK': 'RUC',
    # Forwards
    'FF': 'FWD', 'CHF': 'FWD', 'FPL': 'FWD', 'FPR': 'FWD',
    'HFFL': 'FWD', 'HFFR': 'FWD',
    # Defenders
    'FB': 'DEF', 'CHB': 'DEF', 'BPL': 'DEF', 'BPR': 'DEF',
    'HBFL': 'DEF', 'HBFR': 'DEF',
    # Substitute — minimal TOG, keep isolated
    'SUB': 'UTIL', 'UTIL': 'UTIL',
}

stats['position_group'] = stats['player_position'].map(position_map).fillna('UTIL')
print(stats['position_group'].value_counts())

position_group
MID     3888
DEF     2592
FWD     2592
RUC      432
UTIL     432
Name: count, dtype: int64


In [41]:
print(stats[stats['position_group'] == 'UTIL']['player_position'].value_counts())

player_position
SUB    432
Name: count, dtype: int64


### `won_match` flag

In [42]:
stats['won_match'] = (stats['player_team'] == stats['match_winner']).astype(int)

In [43]:
stats['won_match'].value_counts()

won_match
0    5037
1    4899
Name: count, dtype: int64

### `match_round` → integer

In [44]:
stats['match_round'].unique()

array(['Opening Round', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10',
       '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21',
       '22', '23', '24', 'Finals Week 1', 'Semi Finals',
       'Preliminary Finals', 'Grand Final'], dtype=object)

In [45]:
def parse_round(r):
    r = str(r).strip()
    if r == 'Opening Round':
        return 0
    if r.isdigit():
        return int(r)
    finals_map = {
        'Finals Week 1':       25,
        'Semi Finals':         26,
        'Preliminary Finals':  27,
        'Grand Final':         28,
    }
    return finals_map.get(r, -1)

stats['round_number'] = stats['match_round'].apply(parse_round)
print(stats['round_number'].value_counts().sort_index())

round_number
0     184
1     414
2     368
3     368
4     414
5     368
6     368
7     414
8     414
9     414
10    414
11    414
12    322
13    368
14    276
15    276
16    414
17    414
18    414
19    414
20    414
21    414
22    414
23    414
24    414
25    184
26     92
27     92
28     46
Name: count, dtype: int64


### `match_weather_type` → one-hot

In [46]:
stats['match_weather_type'].unique()

array(['RAIN', 'MOSTLY_SUNNY', 'SUNNY', 'OVERCAST', 'THUNDERSTORMS',
       'WINDY'], dtype=object)

In [47]:
weather_dummies = pd.get_dummies(stats['match_weather_type'], prefix='weather', dtype=int)
stats = pd.concat([stats, weather_dummies], axis=1)
stats.drop('match_weather_type', axis=1, inplace=True)

### Quick sanity check

In [48]:
print(stats[['player_position', 'position_group', 'won_match', 'round_number']].head(10))
print("\nPosition group distribution:\n", stats['position_group'].value_counts())
print("\nWeather columns added:", [c for c in stats.columns if c.startswith('weather_')])

  player_position position_group  won_match  round_number
0              FB            DEF          0             0
1              RK            RUC          0             0
2             CHB            DEF          0             0
3             INT            MID          1             0
4             BPR            DEF          1             0
5            HFFL            FWD          0             0
6              RK            RUC          1             0
7             SUB           UTIL          0             0
8            HBFR            DEF          1             0
9              RR            MID          0             0

Position group distribution:
 position_group
MID     3888
DEF     2592
FWD     2592
RUC      432
UTIL     432
Name: count, dtype: int64

Weather columns added: ['weather_MOSTLY_SUNNY', 'weather_OVERCAST', 'weather_RAIN', 'weather_SUNNY', 'weather_THUNDERSTORMS', 'weather_WINDY']


## 1.4 Derive Per-Match Context Features
These give context to raw stats:

| New Feature             | Formula                                                                  |
|--------------------|-------------------------------------------------------------------------|
| `is_home`    | `player_team == match_home_team` |
| `match_closeness`       | `abs(match_margin)` — how close the game was |
| `team_score` | If home: `match_home_team_score`, else `match_away_team_score`|
| `opponent_score`        | Inverse of above|

In [49]:
if set(stats['player_team'].unique()) == set(stats['match_home_team'].unique()):
    print(True)
else:
    print(False)

True


In [50]:
stats['is_home'] = (stats['player_team'] == stats['match_home_team']).astype(int)

In [51]:
stats['is_home'].unique()

array([0, 1])

In [52]:
stats['match_closeness'] = stats['match_margin'].abs()
stats['team_score'] = stats.apply(
    lambda row: row['match_home_team_score'] if row['is_home'] == 1 else row['match_away_team_score'],
    axis=1
)
stats['opponent_score'] = stats.apply(
    lambda row: row['match_away_team_score'] if row['is_home'] == 1 else row['match_home_team_score'],
    axis=1
)

In [53]:
stats.to_csv("../data/processed/stats.csv", index=False)